# Organize data for each event w/ isolates resequenced w/ PacBio HiFi

# import statements

In [10]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [11]:
import os

In [12]:
import glob

#### Pandas Viewing Settings

In [13]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

# Read in TBP22 resequenced isolates (Selected from `TGENSR` dataset)

In [14]:
Repo_MainDir = "../.."
Repo_DataDir = f"{Repo_MainDir}/Data"

TBP22_GCEVerf_Metadata_Dir = f"{Repo_DataDir}/TBP22.22CI.GCEVerfIsolates.Metadata" 

TBP22_22CI_HybridAsmQCStats_TSV_PATH                = f"{TBP22_GCEVerf_Metadata_Dir}/250801.TBP22.22CI.GCEVerfIsolates.HybridAsmQCStats.V1.tsv"

TBP22_Final_QCPass_22CI_Input_AsmAndReads_Paths_TSV = f"{TBP22_GCEVerf_Metadata_Dir}/250801.TBP22.22CI.GCEVerfIsolates.Asm_LR_SR.InputPATHs.tsv"


In [1]:
!pwd

/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb-GeneConv/mtb-GE-analysis/Analysis/7_TGEN_GCVerf_Part2_OrgDataForSelectedIsolates


### TBP22 Hybrid Complete Genome Assembly QC Stats 

In [15]:
TBP22_22CI_AsmQC_DF = pd.read_csv(TBP22_22CI_HybridAsmQCStats_TSV_PATH,
                                  sep ="\t")
TBP22_22CI_AsmQC_DF.shape

(22, 19)

In [16]:
TBP22_22CI_AsmQC_DF.head(1)

,SampleID,numContigs_Complete,circContig_Length,circContig_Cov,Flye_EstimatedCov,Flye_ReadLen_N50,Flye_ReadLen_N90,Lineage_Asm,Lineage_AsmPP,PrimaryLineage_Asm,Dataset_Tag,SR_SRA_RunAcc,SeqReason,PB_SeqRunName,EventID,Event_Gene(s),Event_Relationship,TBP_SampleID,TGEN_SampleID
0,TB3706,1,4412093,127,129,4204,2724,lineage2.2.1,lineage2.2.1,lineage2,TBPortals2022,SRR10397096,ReseqToVerfGCE,P7529,Event_006,"Rv0979c,rpmF,PE_PGRS18",Outside_Event,TB3706,DNA621


##### Create dictionaries for mapping varying sampleIDs for the same isolates

In [17]:
TGEN_To_TBP_SampleID_Dict = dict(TBP22_22CI_AsmQC_DF[['TGEN_SampleID', 'TBP_SampleID']].values)
TBP_To_TGEN_SampleID_Dict = dict(TBP22_22CI_AsmQC_DF[['TBP_SampleID', 'TGEN_SampleID']].values)
TGEN_To_SR_SRA_RunAcc_Dict = dict(TBP22_22CI_AsmQC_DF[['TGEN_SampleID', 'SR_SRA_RunAcc']].values)
TBP_To_SR_SRA_RunAcc_Dict = dict(TBP22_22CI_AsmQC_DF[['TBP_SampleID', 'SR_SRA_RunAcc']].values)


### TBP22 - Assembly + Illumina WGS + PacBio WGS File Paths

In [18]:
TBP22_22CI_EventVerf_AsmAndRead_PATHS_V1_DF = pd.read_csv(TBP22_Final_QCPass_22CI_Input_AsmAndReads_Paths_TSV,
                                                                  sep ="\t")

TBP22_22CI_EventVerf_AsmAndRead_PATHS_V1_DF.shape

(22, 8)

In [19]:
TBP22_22CI_EventVerf_AsmAndRead_PATHS_V1_DF.head(1)

,SampleID,TGEN_SampleID,SRA_RunAcc_SR,Dataset_Tag,SeqReason,Illumina_PE_FQs_PATH,PacBio_FQ_PATH,HybridAsm_FA_PATH
0,TB6733,DNA0428,SRR10379945,TBPortals_2022_PassQC,PutativeRecomb,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...


# Define Isolate and event metadata for resequenced isolates

## Define metadata for each event that is targeted for validation

In [20]:

TGENSR_SelectedIsolates_GCEventVerf = ['TB8073', 'TB6755', 'TB6973', 'TB6778',
                                       'TB3572', 'TB6786', 'TB6552', 'TB6599',
                                       'TB6765', 'TB6977', 'TB6733', 'TB3898',
                                       'TB3706', 'TB3256', 'TB3305', 'TB6976',
                                       'TB6807', 'TB4414', 'TB7340', 'TB6846',
                                       'TB6596', 'TB7044']

dictOf_TGEN_GCEventVerf_TBP_ID_Mappings = {
    "Event_001": {"Target": "TB6733", "Control": "TB3898"},
    "Event_003": {"Target": "TB6599", "Control": "TB6977"},
    "Event_006": {"Target": "TB3305", "Control": "TB3706"},
    "Event_007": {"Target": "TB6755", "Control": "TB7044"},
    "Event_010": {"Target": "TB6552", "Control": "TB6765"},
    "Event_011": {"Target": "TB6778", "Control": "TB6973"},
    "Event_013": {"Target": "TB6786", "Control": "TB3256"},
    "Event_019": {"Target": "TB6977", "Control": "TB6976"},
    "Event_021": {"Target": "TB3572", "Control": "TB6976"},
    "Event_022": {"Target": "TB6596", "Control": "TB8073"},
    "Event_024": {"Target": "TB7340", "Control": "TB6807"},
    "Event_025": {"Target": "TB6846", "Control": "TB4414"},
}

# Convert to DataFrame
TGENSR_Events_ReseqIsolates_Info_DF = pd.DataFrame.from_dict(dictOf_TGEN_GCEventVerf_TBP_ID_Mappings, orient='index')
TGENSR_Events_ReseqIsolates_Info_DF = TGENSR_Events_ReseqIsolates_Info_DF.rename(columns={"Target": "Verification_IsolateID", "Control": "Control_IsolateID"})
TGENSR_Events_ReseqIsolates_Info_DF.index.name = "EventID"
TGENSR_Events_ReseqIsolates_Info_DF.reset_index(inplace=True)

TGENSR_Events_ReseqIsolates_Info_DF["Verfication_SRWGS_RunID"] = TGENSR_Events_ReseqIsolates_Info_DF["Verification_IsolateID"].map(TBP_To_SR_SRA_RunAcc_Dict)
TGENSR_Events_ReseqIsolates_Info_DF["Control_SRWGS_RunID"]     = TGENSR_Events_ReseqIsolates_Info_DF["Control_IsolateID"].map(TBP_To_SR_SRA_RunAcc_Dict)


print(TGENSR_Events_ReseqIsolates_Info_DF.shape)

# Create EventID → Target dictionary
EventID_to_TargetIsolateID = TGENSR_Events_ReseqIsolates_Info_DF.set_index("EventID")["Verification_IsolateID"].to_dict()

# Create EventID → Control dictionary
EventID_to_ControlIsolateID = TGENSR_Events_ReseqIsolates_Info_DF.set_index("EventID")["Control_IsolateID"].to_dict()


(12, 5)


## Define table of isolateIDs to the events they are validating

In [21]:

# Build long-form list of event-role assignments
IsolateToEvent_records = []
for event_id, pair in dictOf_TGEN_GCEventVerf_TBP_ID_Mappings.items():
    IsolateToEvent_records.append({
        "SampleID": pair["Target"],
        "EventID": event_id,
        "Role": "Verification",
        "Description": f"Verification for {event_id}"
    })
    IsolateToEvent_records.append({
        "SampleID": pair["Control"],
        "EventID": event_id,
        "Role": "Control",
        "Description": f"Control for {event_id}"
    })

# Create long-form DataFrame
IsolateToEvent_V1_DF = pd.DataFrame(IsolateToEvent_records)

# Aggregate to wide format: one row per SampleID
IsolateToEvent_V2_DF = (
    IsolateToEvent_V1_DF
    .groupby("SampleID")
    .agg({
        "EventID": lambda x: ";".join(sorted(set(x))),
        "Role": lambda x: ";".join(sorted(set(x))),
        "Description": lambda x: ";".join(sorted(x))
    })
    .reset_index()
    .rename(columns={"EventID": "EventIDs", "Role": "Roles", "Description": "All_Descriptions"})
)

TBP22_IsolateToGCE_V2_DF = IsolateToEvent_V2_DF.sort_values(["EventIDs", "Roles"]).reset_index(drop=True)

TBP22_IsolateToGCE_V2_DF.shape

(22, 4)

## Inspect summary tables describing the Resequenced isolates and which events they are related to

In [22]:
TGENSR_Events_ReseqIsolates_Info_DF

,EventID,Verification_IsolateID,Control_IsolateID,Verfication_SRWGS_RunID,Control_SRWGS_RunID
0,Event_001,TB6733,TB3898,SRR10379945,SRR10397263
1,Event_003,TB6599,TB6977,SRR10379958,SRR10380218
2,Event_006,TB3305,TB3706,SRR10397175,SRR10397096
3,Event_007,TB6755,TB7044,SRR10379935,SRR10380192
4,Event_010,TB6552,TB6765,SRR10380108,SRR10379924
5,Event_011,TB6778,TB6973,SRR10380252,SRR10380223
6,Event_013,TB6786,TB3256,SRR10380244,SRR10397205
7,Event_019,TB6977,TB6976,SRR10380218,SRR10380219
8,Event_021,TB3572,TB6976,SRR10397163,SRR10380219
9,Event_022,TB6596,TB8073,SRR10379961,SRR10380026


In [23]:
TBP22_IsolateToGCE_V2_DF

,SampleID,EventIDs,Roles,All_Descriptions
0,TB3898,Event_001,Control,Control for Event_001
1,TB6733,Event_001,Verification,Verification for Event_001
2,TB6599,Event_003,Verification,Verification for Event_003
3,TB6977,Event_003;Event_019,Control;Verification,Control for Event_003;Verification for Event_019
4,TB3706,Event_006,Control,Control for Event_006
5,TB3305,Event_006,Verification,Verification for Event_006
6,TB7044,Event_007,Control,Control for Event_007
7,TB6755,Event_007,Verification,Verification for Event_007
8,TB6765,Event_010,Control,Control for Event_010
9,TB6552,Event_010,Verification,Verification for Event_010


# Output summary tables - mapping IsolateIDs to their role in verifying specific events

In [24]:
TBP22_GCEVerf_Metadata_Dir = f"{Repo_DataDir}/TBP22.22CI.GCEVerfIsolates.Metadata" 
!mkdir $TBP22_GCEVerf_Metadata_Dir

mkdir: cannot create directory ‘../../Data/TBP22.22CI.GCEVerfIsolates.Metadata’: File exists


In [25]:
TBP22_GCEvent_To_IsolateIDs_TSV = f"{TBP22_GCEVerf_Metadata_Dir}/250801.TBP22.22CI.TGENSR_Reseq.GCEvent_To_IsolateIDs.tsv"

TBP22_IsolateID_To_GCEvents_TSV = f"{TBP22_GCEVerf_Metadata_Dir}/250801.TBP22.22CI.TGENSR_Reseq.IsolateID_To_GCEvents.tsv"


In [26]:
TGENSR_Events_ReseqIsolates_Info_DF.to_csv(TBP22_GCEvent_To_IsolateIDs_TSV, sep ="\t", index=False)

In [27]:
TBP22_IsolateToGCE_V2_DF.to_csv(TBP22_IsolateID_To_GCEvents_TSV, sep ="\t", index=False)

### inspect output TSVs

In [28]:
!head $TBP22_GCEvent_To_IsolateIDs_TSV

EventID	Verification_IsolateID	Control_IsolateID	Verfication_SRWGS_RunID	Control_SRWGS_RunID
Event_001	TB6733	TB3898	SRR10379945	SRR10397263
Event_003	TB6599	TB6977	SRR10379958	SRR10380218
Event_006	TB3305	TB3706	SRR10397175	SRR10397096
Event_007	TB6755	TB7044	SRR10379935	SRR10380192
Event_010	TB6552	TB6765	SRR10380108	SRR10379924
Event_011	TB6778	TB6973	SRR10380252	SRR10380223
Event_013	TB6786	TB3256	SRR10380244	SRR10397205
Event_019	TB6977	TB6976	SRR10380218	SRR10380219
Event_021	TB3572	TB6976	SRR10397163	SRR10380219


In [29]:
!head $TBP22_IsolateID_To_GCEvents_TSV

SampleID	EventIDs	Roles	All_Descriptions
TB3898	Event_001	Control	Control for Event_001
TB6733	Event_001	Verification	Verification for Event_001
TB6599	Event_003	Verification	Verification for Event_003
TB6977	Event_003;Event_019	Control;Verification	Control for Event_003;Verification for Event_019
TB3706	Event_006	Control	Control for Event_006
TB3305	Event_006	Verification	Verification for Event_006
TB7044	Event_007	Control	Control for Event_007
TB6755	Event_007	Verification	Verification for Event_007
TB6765	Event_010	Control	Control for Event_010


# Extra